# 4. Generación de Imágenes

Crear imágenes desde texto.

## Setup

Instalar SDK.

In [ ]:
%pip install --upgrade openai pillow requests

## API Key

Usar variable de entorno.

In [ ]:
import os
import base64
from pathlib import Path
from getpass import getpass
from IPython.display import Image, display
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

client = OpenAI()
print("Cliente listo")

## 4.1 Modelos disponibles

Actuales:
- `gpt-image-2`
- `gpt-image-1.5`
- `gpt-image-1`

Legado:
- `dall-e-3`
- `dall-e-2`

In [ ]:
IMAGE_MODEL = os.getenv("IMAGE_MODEL", "gpt-image-2")
print("Modelo:", IMAGE_MODEL)

## 4.2 Parámetros de la API

Claves:
- `model`
- `prompt`
- `size`
- `quality`
- `background`
- `output_format`

In [ ]:
prompt = """
Ilustración limpia para una presentación:
un robot pequeño ayudando a una persona a organizar tickets de soporte,
estilo moderno, fondo claro, sin texto.
"""

result = client.images.generate(
    model=IMAGE_MODEL,
    prompt=prompt,
    size="1024x1024",
    quality="medium",
    background="auto",
    output_format="png",
)

image_base64 = result.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

output_path = Path("imagen_generada.png")
output_path.write_bytes(image_bytes)

print(f"Guardado en: {output_path}")
display(Image(filename=str(output_path)))

## Variante: tamaño vertical

Útil para posts o posters.

In [ ]:
prompt_vertical = """
Poster vertical minimalista sobre inteligencia artificial responsable.
Usar formas geométricas simples, luz suave, sin texto.
"""

result = client.images.generate(
    model=IMAGE_MODEL,
    prompt=prompt_vertical,
    size="1024x1536",
    quality="medium",
    background="auto",
    output_format="png",
)

image_base64 = result.data[0].b64_json
output_path = Path("imagen_vertical.png")
output_path.write_bytes(base64.b64decode(image_base64))

display(Image(filename=str(output_path)))

## Variante: fondo transparente

Útil para íconos/logos.

In [ ]:
prompt_icon = """
Ícono simple de una nube con un rayo pequeño.
Estilo vectorial, limpio, centrado, sin texto.
"""

result = client.images.generate(
    model=IMAGE_MODEL,
    prompt=prompt_icon,
    size="1024x1024",
    quality="medium",
    background="transparent",
    output_format="png",
)

image_base64 = result.data[0].b64_json
output_path = Path("icono_transparente.png")
output_path.write_bytes(base64.b64decode(image_base64))

display(Image(filename=str(output_path)))

## 4.3 Descarga y almacenamiento

Función reusable.

In [ ]:
def save_b64_image(b64_json: str, filename: str) -> Path:
    path = Path(filename)
    path.write_bytes(base64.b64decode(b64_json))
    return path


def generate_image(prompt: str, filename: str, size: str = "1024x1024", quality: str = "medium") -> Path:
    result = client.images.generate(
        model=IMAGE_MODEL,
        prompt=prompt,
        size=size,
        quality=quality,
        output_format="png",
    )
    return save_b64_image(result.data[0].b64_json, filename)


path = generate_image(
    prompt="Ilustración flat de un equipo revisando un dashboard de métricas, sin texto.",
    filename="dashboard_ai.png",
)

display(Image(filename=str(path)))

## DALL·E 3 legado

Solo si querés compatibilidad antigua.

In [ ]:
dalle_model = "dall-e-3"

try:
    result = client.images.generate(
        model=dalle_model,
        prompt="Una taza de café futurista sobre un escritorio minimalista.",
        size="1024x1024",
        quality="standard",
        response_format="url",
    )

    print(result.data[0].url)
    print("Abrir la URL para ver la imagen.")

except Exception as e:
    print("Ejemplo DALL·E no ejecutado o no soportado en tu cuenta/modelo.")
    print(e)

## Responses API con herramienta de imagen

Útil para flujos multi-turn.

In [ ]:
response = client.responses.create(
    model=os.getenv("OPENAI_MODEL", "gpt-5.5"),
    input="Genera una imagen de un gato gris abrazando una nutria con bufanda naranja.",
    tools=[{"type": "image_generation", "action": "generate"}],
)

image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]

if image_data:
    path = save_b64_image(image_data[0], "responses_image.png")
    display(Image(filename=str(path)))
else:
    print(response.output_text)

## 4.4 Buenas prácticas

- Prompt visual y específico.
- Indicar estilo.
- Pedir “sin texto” si no querés texto.
- Definir formato y orientación.
- Guardar prompt usado.
- Revisar derechos y políticas.

## Ejercicio

Crear 3 variantes:
1. ícono,
2. portada,
3. imagen para slide.

In [ ]:
ejercicios = {
    "icono_api.png": "Ícono vectorial de una API conectando dos sistemas, sin texto.",
    "portada_curso.png": "Portada moderna para un curso de OpenAI API con Python, sin texto.",
    "slide_visual.png": "Imagen 16:9 conceptual de IA ayudando a programadores, sin texto.",
}

for filename, prompt in ejercicios.items():
    path = generate_image(prompt, filename)
    print(path)